In [19]:
import joblib
import pandas as pd

model = joblib.load("/heart_attack_prediction_model.pkl")

print("Heart Attack Risk Prediction Model Loaded Successfully")
print("Model:", type(model).__name__)

Heart Attack Risk Prediction Model Loaded Successfully
Model: SVC


In [20]:
# Sample patient data
sample_patient = pd.DataFrame([{
    'age': 55,
    'sex': 1,
    'cp': 2,
    'trtbps': 140,
    'chol': 250,
    'fbs': 0,
    'restecg': 1,
    'thalachh': 150,
    'exng': 0,
    'oldpeak': 1.0,
    'slp': 1,
    'caa': 0,
    'thall': 2
}])

print("Patient data:")
display(sample_patient)

Patient data:


,age,sex,cp,trtbps,chol,fbs,restecg,thalachh,exng,oldpeak,slp,caa,thall
0,55,1,2,140,250,0,1,150,0,1.0,1,0,2


In [21]:
# Check the number of features expected by the trained model
print("Model expects features:", model.n_features_in_)
print("Sample has features:", sample_patient.shape[1])

Model expects features: 30
Sample has features: 13


In [22]:
# Preprocess sample patient using the training data

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('/heart.csv')

# Separate features and target
X = df.drop('output', axis=1)
y = df['output']

# Recreate the same train/validation/test split
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.40,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

# Feature groups
continuous_features = [
    'age',
    'trtbps',
    'chol',
    'thalachh',
    'oldpeak'
]

categorical_features = [
    'sex',
    'cp',
    'fbs',
    'restecg',
    'exng',
    'slp',
    'caa',
    'thall'
]

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), continuous_features),
        ('cat', OneHotEncoder(
            handle_unknown='ignore',
            sparse_output=False
        ), categorical_features)
    ]
)

# IMPORTANT: fit using training data, NOT the single patient
preprocessor.fit(X_train)

# Transform the sample patient
sample_processed = preprocessor.transform(sample_patient)

print("Original sample shape:", sample_patient.shape)
print("Processed sample shape:", sample_processed.shape)

Original sample shape: (1, 13)
Processed sample shape: (1, 30)


In [23]:
# Predict heart attack risk

print("Input shape for prediction:", sample_processed.shape)
print("Model expects:", model.n_features_in_, "features")

prediction = model.predict(sample_processed)[0]

if prediction == 1:
    print("Prediction: Heart Attack Risk")
else:
    print("Prediction: No Heart Attack Risk")

Input shape for prediction: (1, 30)
Model expects: 30 features
Prediction: Heart Attack Risk


## Conclusion

The trained supervised learning model successfully predicts myocardial infarction risk from patient attributes. The input data is transformed using the same preprocessing pipeline used during model training, ensuring compatibility with the trained SVM model.

The final demonstration confirms that the complete workflow, from patient input to risk prediction, is functioning successfully.